# Machine learning pipelines and beyond
* lesson 21 covered the absolute basics of ml: one classifier, one train/test split, fit/predict, accuracy_score, and the idea of overfitting
* that gets you a working model, but its not really how ml is done in practice - real workflows need to handle preprocessing properly, be more confident that the score they report isnt just luck, and actually pick good settings instead of guessing
* this lesson is about that "next level" of practical scikit-learn: feature scaling, `Pipeline`, cross-validation, `GridSearchCV`, and a first, tiny neural network with `MLPClassifier`

## feature scaling - why it matters
* a lot of ml algorithms are secretly doing distance/geometry math under the hood (e.g. k-nearest-neighbors literally measures distance between points, neural networks multiply inputs by weights)
* if one feature is "age in years" (values like 20-80) and another is "income in dollars" (values like 20000-200000), the income feature will completely dominate any distance calculation just because its numbers are bigger - not because its actually more important
* `StandardScaler` fixes this by rescaling every feature to have mean 0 and standard deviation 1, so all features are on a comparable numeric scale
* **very important gotcha**: we `.fit_transform()` the scaler on the TRAINING data only, then `.transform()` (no fit!) the test data with those same, already-learned train-set stats - if we fit on the test set too we would leak information about the test set into our "training" process, which makes our evaluation dishonestly optimistic

In [1]:
# lets build a small toy dataset with two features that are intentionally on very different scales
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# make_classification generates a synthetic classification dataset - no download needed, fully offline
# n_features=4 gives us 4 numeric columns, n_classes=2 makes it a simple yes/no style problem
X, y = make_classification(
    n_samples=300,        # 300 rows total
    n_features=4,         # 4 numeric input columns
    n_informative=3,      # 3 of those columns actually carry signal
    n_redundant=0,        # 0 columns that are just linear combinations of the informative ones (keeps this simple)
    n_classes=2,           # a binary classification problem
    random_state=42,      # random_state makes this reproducible - same "random" data every run
)

# lets deliberately blow up the scale of one column to make the scaling problem obvious
X[:, 0] = X[:, 0] * 1000  # now column 0 has values in the thousands, while the others stay small

# split BEFORE scaling - the split itself doesnt care about scale, but the order matters for what comes next
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("before scaling, column 0 (train) ranges roughly from", X_train[:, 0].min(), "to", X_train[:, 0].max())
print("before scaling, column 1 (train) ranges roughly from", X_train[:, 1].min(), "to", X_train[:, 1].max())
print("--> column 0 completely dwarfs column 1 in raw magnitude, even though both might matter equally for prediction")

# now the scaler: fit_transform on train (learns mean/std from train AND applies it),
# then only transform (no fit!) on test, reusing the train-set mean/std
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # LEARNS mean/std from training data, then applies it
X_test_scaled = scaler.transform(X_test)          # REUSES those same train stats, does NOT relearn from test data

print()
print("after scaling, column 0 (train) mean is roughly", round(X_train_scaled[:, 0].mean(), 4), "and std is roughly", round(X_train_scaled[:, 0].std(), 4))
print("after scaling, column 1 (train) mean is roughly", round(X_train_scaled[:, 1].mean(), 4), "and std is roughly", round(X_train_scaled[:, 1].std(), 4))
print("--> both columns are now on the same comparable scale (mean ~0, std ~1)")

# NEVER do scaler.fit_transform(X_test) - that would let the model peek at test-set statistics before evaluation, a classic form of data leakage

before scaling, column 0 (train) ranges roughly from -4660.8576373898995 to 4304.434533332298
before scaling, column 1 (train) ranges roughly from -4.845551450606566 to 3.7108324816597342
--> column 0 completely dwarfs column 1 in raw magnitude, even though both might matter equally for prediction

after scaling, column 0 (train) mean is roughly -0.0 and std is roughly 1.0
after scaling, column 1 (train) mean is roughly -0.0 and std is roughly 1.0
--> both columns are now on the same comparable scale (mean ~0, std ~1)


## Pipeline - chaining preprocessing and model into ONE object
* doing `scaler.fit_transform(X_train)` and `scaler.transform(X_test)` by hand works, but its easy to mess up in a bigger project - forget a step, apply steps in a different order, or (worst case) accidentally fit the scaler on test/production data
* `Pipeline` solves this by bundling several steps (e.g. "scale, then classify") into a single estimator object - you call `.fit()` and `.predict()` on the PIPELINE exactly like you would on a plain model, and it automatically applies every step in the right order, with the right fit/transform rules, every single time
* this is one of the most important "grows up" habits when moving from toy ml scripts to a real, maintainable ml workflow

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# a Pipeline is just a list of (name, step) tuples, applied in order
# step 1: "scaler" - a StandardScaler, step 2: "knn" - the actual classifier (k-nearest-neighbors)
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),           # this step scales the features
    ("knn", KNeighborsClassifier(n_neighbors=5)),  # this step does the actual classification
])

# now we just .fit() the WHOLE pipeline on the raw, unscaled training data
# internally it does: fit_transform the scaler on X_train, THEN fit the knn on the scaled result
knn_pipeline.fit(X_train, y_train)

# .predict() on the pipeline automatically transforms X_test with the ALREADY-FITTED scaler first, then predicts
# we never had to call scaler.transform() ourselves - the pipeline remembers to do it, in the right order, every time
predictions = knn_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"pipeline (scaler + knn) accuracy on the test set: {accuracy:.3f}")

# this is exactly the kind of thing that would be dangerously easy to get wrong by hand once your project grows:
# e.g. accidentally applying the RAW (unscaled) test features to a model trained on SCALED data would silently give garbage predictions
# the pipeline makes that whole class of bug basically impossible, since scaling and predicting are glued together

pipeline (scaler + knn) accuracy on the test set: 0.853


## cross-validation - is our score just lucky?
* a single `train_test_split` gives us one score, based on whichever rows randomly happened to end up in the test set - a slightly "easy" or "hard" split can nudge that score up or down without the model actually being better or worse
* cross-validation fixes this by splitting the data into multiple different train/test combinations and scoring the model on EACH one, giving us a whole distribution of scores instead of a single lucky/unlucky number
* `cross_val_score(pipeline, X, y, cv=5)` splits the data into 5 "folds", trains on 4 of them and tests on the 5th, five times over (a different fold left out for testing each time) - note it takes the raw `X`/`y` (not X_train/X_test) since it does all the splitting internally
* we then look at both the MEAN (our best single estimate of "how good is this model") and the STANDARD DEVIATION (how much that score wobbles between different splits - a big std means the model's performance is unstable and we should be a bit less confident in any single score)

In [3]:
from sklearn.model_selection import cross_val_score

# cross_val_score handles the splitting internally - we pass it the FULL X, y (not a pre-split X_train/X_test)
# cv=5 means "5-fold cross-validation": 5 different train/test combinations, 5 resulting scores
cv_scores = cross_val_score(knn_pipeline, X, y, cv=5)

print("all 5 cross-validation scores:", np.round(cv_scores, 3))
print(f"mean accuracy across folds: {cv_scores.mean():.3f}")
print(f"standard deviation across folds: {cv_scores.std():.3f}")

# why this matters: the mean gives us a much more reliable "typical" score than any single train/test split
# and the std tells us how much we should trust that mean - a std of 0.15 means "this model's score bounces around a LOT depending on the split", which is worth knowing before you tell your boss "our model is 90% accurate"!
# note: cross_val_score correctly re-fits the WHOLE pipeline (scaler included) fresh on each fold's training portion, so there is still no leakage between that fold's train and test data

all 5 cross-validation scores: [0.817 0.883 0.783 0.8   0.867]
mean accuracy across folds: 0.830
standard deviation across folds: 0.039


## hyperparameter tuning with GridSearchCV
* a "hyperparameter" is a setting YOU choose before training starts - like "how many neighbors should knn look at" (`n_neighbors`) or "how deep should a decision tree be allowed to grow" (`max_depth`)
* this is different from what the model LEARNS from the data itself (e.g. the actual decision boundaries or tree splits) - hyperparameters are the knobs, learned parameters are the result of turning them on some data
* trying every combination by hand (train with n_neighbors=3, check the score, train with n_neighbors=5, check the score, ...) works but is tedious and error-prone
* `GridSearchCV` automates this: give it a "grid" of hyperparameter values to try, it trains and cross-validates a model for EVERY combination, and reports which combination scored best (`.best_params_`) and what that best score was (`.best_score_`)
* we keep our grid tiny here (just a few values, cv=3) so it runs fast - in a real project you might search a much bigger grid, but the idea is identical

In [4]:
from sklearn.model_selection import GridSearchCV

# a "grid" is just a dict: {hyperparameter name inside the pipeline --> list of values to try}
# note the "knn__" prefix - that tells the pipeline "this hyperparameter belongs to the step named 'knn'"
param_grid = {
    "knn__n_neighbors": [3, 5, 9],  # try 3 different neighborhood sizes
}

# GridSearchCV wraps our pipeline, tries every value in param_grid, and cross-validates (cv=3) each one
grid_search = GridSearchCV(knn_pipeline, param_grid, cv=3)
grid_search.fit(X, y)  # this trains+evaluates 3 hyperparameter values x 3 cv folds = 9 small training runs total, still fast

print("best n_neighbors found:", grid_search.best_params_)
print(f"best cross-validated accuracy: {grid_search.best_score_:.3f}")

# grid_search.best_estimator_ is a fully-fitted pipeline already using the winning hyperparameters - ready to .predict() with directly
best_predictions = grid_search.best_estimator_.predict(X_test)
print(f"best pipeline accuracy on our original held-out test set: {accuracy_score(y_test, best_predictions):.3f}")

best n_neighbors found: {'knn__n_neighbors': 3}
best cross-validated accuracy: 0.820
best pipeline accuracy on our original held-out test set: 0.920


## a tiny neural network with MLPClassifier
* a neural network is, in plain words, layers of simple math units ("neurons") stacked on top of each other - each neuron combines its inputs (a weighted sum) and passes the result through a small non-linear transformation, and the whole network's weights get nudged, bit by bit, during training to reduce prediction errors
* scikit-learn ships a basic one out of the box: `MLPClassifier` ("Multi-Layer Perceptron") - its a small, simple, CPU-friendly neural network, perfect for learning the concept and for small/medium tabular datasets like ours
* **its important to be honest about scope here**: REAL modern deep learning (large image/language models, trained on GPUs/TPUs) uses dedicated frameworks instead - **PyTorch**, **TensorFlow** and **JAX** are the big three, and they add things `MLPClassifier` doesnt have: automatic differentiation (automatically computing gradients for arbitrarily complex, custom network architectures), GPU/TPU acceleration (running the same math on specialized hardware, often 10-100x faster than a CPU), and huge pretrained model ecosystems (download a model someone else already spent months training and fine-tune it, instead of starting from zero)
* we wont install or run any of those here - `MLPClassifier` is our friendly, offline, CPU-only starting point to get the CONCEPT of a neural network, not a production deep learning setup

In [5]:
from sklearn.neural_network import MLPClassifier
import warnings

# an MLPClassifier pipeline, same scaling step as before - neural networks are VERY sensitive to feature scale, even more so than knn
mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    # hidden_layer_sizes=(8,) means: one hidden layer with 8 neurons - tiny and fast on purpose
    # max_iter caps how many training passes it does, so this stays quick and doesnt need convergence-tuning for our demo
    ("mlp", MLPClassifier(hidden_layer_sizes=(8,), max_iter=2000, random_state=42)),
])

# small networks on small data can print a harmless "didnt fully converge" warning - we suppress it here, it doesnt affect our demo point
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mlp_pipeline.fit(X_train, y_train)

mlp_predictions = mlp_pipeline.predict(X_test)
mlp_accuracy = accuracy_score(y_test, mlp_predictions)

print(f"knn pipeline accuracy:  {accuracy:.3f}")
print(f"mlp (tiny neural net) accuracy: {mlp_accuracy:.3f}")
print()
print("on a small, simple dataset like this one, a tiny neural net often does about as well as knn (or even a bit worse) -")
print("neural networks really start to shine on much bigger, messier datasets (images, text, audio) where they can learn complex patterns knn/trees struggle with")

knn pipeline accuracy:  0.853
mlp (tiny neural net) accuracy: 0.867

on a small, simple dataset like this one, a tiny neural net often does about as well as knn (or even a bit worse) -
neural networks really start to shine on much bigger, messier datasets (images, text, audio) where they can learn complex patterns knn/trees struggle with


## quick comparison to other languages
* scikit-learn's `Pipeline` + `GridSearchCV` combo is conceptually very close to R's `tidymodels`/`caret` workflow (define a preprocessing "recipe", attach a model, wrap it all in a tuning grid) - same idea, different syntax
* it's also basically the same idea, and even the same NAME, as Java/Scala Spark MLlib's `Pipeline` API - "chain preprocessing stages and a model into one reusable, fit-able object" is such a common, important pattern that it converged on nearly the same design independently in multiple ecosystems
* the reason so many serious ml ecosystems converge on this "compose everything into one pipeline object" pattern is that manually keeping preprocessing steps in sync between training and inference (production) is one of THE most common, painful real-world ml bugs - bundling it all into one object makes that class of mistake structurally much harder to make
* deep learning specifically is overwhelmingly python-first - PyTorch, TensorFlow and JAX are all designed and developed primarily as python libraries; other languages get bindings/ports (e.g. libtorch lets you run PyTorch models from C++, ONNX Runtime lets you run an exported model for inference from many languages), but almost all serious model TRAINING work happens in python, regardless of what language the final deployed application ends up written in

## Exercises
1. build a `Pipeline` combining a `StandardScaler` and a `KNeighborsClassifier`, then `.fit()` it on the training data and print its accuracy on the test set (you can reuse the `X_train`/`X_test`/`y_train`/`y_test` from this lesson, or build a fresh toy dataset)
2. run `cross_val_score` with `cv=5` on that pipeline (on the full `X`/`y`) and print both the mean and the standard deviation of the resulting scores
3. use `GridSearchCV` to try 3 different values of `n_neighbors` (e.g. 3, 7, 11) and print out the best value found and its cross-validated score
4. train an `MLPClassifier` (wrapped in a `Pipeline` with a `StandardScaler`, just like in this lesson) on the same data, and print its test-set accuracy next to the knn pipeline's accuracy for comparison

In [6]:
# TODO: solve exercise 1 here (Pipeline of StandardScaler + KNeighborsClassifier, fit + accuracy on test set)


# TODO: solve exercise 2 here (cross_val_score with cv=5, print mean and std)


# TODO: solve exercise 3 here (GridSearchCV over 3 values of n_neighbors, print best_params_ and best_score_)


# TODO: solve exercise 4 here (MLPClassifier pipeline, compare its accuracy to the knn pipeline's)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [7]:
# sample solution 1 - Pipeline of StandardScaler + KNeighborsClassifier
pipeline_ex1 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
pipeline_ex1.fit(X_train, y_train)  # fit the whole pipeline (scaling + knn) in one call
predictions_ex1 = pipeline_ex1.predict(X_test)
print("exercise 1 - pipeline accuracy:", round(accuracy_score(y_test, predictions_ex1), 3))

# sample solution 2 - cross_val_score with cv=5
cv_scores_ex2 = cross_val_score(pipeline_ex1, X, y, cv=5)
print("exercise 2 - cv scores:", np.round(cv_scores_ex2, 3))
print("exercise 2 - mean:", round(cv_scores_ex2.mean(), 3), "std:", round(cv_scores_ex2.std(), 3))

# sample solution 3 - GridSearchCV over n_neighbors
grid_ex3 = GridSearchCV(pipeline_ex1, {"knn__n_neighbors": [3, 7, 11]}, cv=3)
grid_ex3.fit(X, y)
print("exercise 3 - best params:", grid_ex3.best_params_)
print("exercise 3 - best score:", round(grid_ex3.best_score_, 3))

# sample solution 4 - MLPClassifier pipeline vs knn pipeline
pipeline_ex4_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(hidden_layer_sizes=(8,), max_iter=2000, random_state=42)),
])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pipeline_ex4_mlp.fit(X_train, y_train)
mlp_predictions_ex4 = pipeline_ex4_mlp.predict(X_test)
print("exercise 4 - knn pipeline accuracy:", round(accuracy_score(y_test, predictions_ex1), 3))
print("exercise 4 - mlp pipeline accuracy:", round(accuracy_score(y_test, mlp_predictions_ex4), 3))

exercise 1 - pipeline accuracy: 0.853
exercise 2 - cv scores: [0.817 0.883 0.783 0.8   0.867]
exercise 2 - mean: 0.83 std: 0.039
exercise 3 - best params: {'knn__n_neighbors': 3}
exercise 3 - best score: 0.82


exercise 4 - knn pipeline accuracy: 0.853
exercise 4 - mlp pipeline accuracy: 0.867


* congratulation you finished this lesson

## what we learned
* why feature scaling matters (algorithms like knn or neural networks are sensitive to features being on very different numeric scales), and how `StandardScaler` fixes it - always fit on train only, then transform on test, never fit on test
* `Pipeline` - chaining preprocessing and a model into one fit/predict-able object, avoiding the classic "scaling applied inconsistently between train/test/production" bug
* cross-validation (`cross_val_score`, `cv=5`) - why a single train/test split can be lucky/unlucky, and how looking at both the mean AND standard deviation of multiple scores gives a more honest, more stable picture of model performance
* hyperparameters vs learned parameters, and using `GridSearchCV` to automatically search a small grid of hyperparameter values and report the best one (`.best_params_`, `.best_score_`)
* a first, tiny neural network with `MLPClassifier`, what a neural network conceptually is (layers of neurons combining/transforming inputs, tuned during training), and how it compares to real production deep learning frameworks (PyTorch, TensorFlow, JAX) that add automatic differentiation, GPU/TPU acceleration and huge pretrained model ecosystems